# Step 2 — Second-level GLM and simple-effect contrasts

Fit the group-level model and write unthresholded z-maps for each simple contrast.

In [1]:
import os
import os.path as op
import re
import sys
from glob import glob
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
import patsy
import yaml
from importlib.resources import files as resource_files
from joblib import Memory, Parallel, delayed
from nilearn.glm.second_level import SecondLevelModel
from patsy.contrasts import ContrastMatrix

for _base in (Path.cwd(), *Path.cwd().parents[:6]):
    for _candidate in (_base, _base / "brain_activation"):
        if (_candidate / "config.yaml").is_file():
            ROOT = _candidate.resolve()
            break
    else:
        continue
    break
else:
    raise FileNotFoundError(
        "Could not find brain_activation/config.yaml. "
        "Open the brain_activation project in Jupyter."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PROJECT = ROOT.parent
RESULTS = ROOT / "results"
artifacts_dir = RESULTS / "artifacts"
res_path = RESULTS / "second_level"
cache_dir = RESULTS / "cachedir"
cache_dir.mkdir(parents=True, exist_ok=True)

def resolve_config_path(path_str):
    p = Path(path_str)
    return p.resolve() if p.is_absolute() else (ROOT / p).resolve()

cfg = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))
PATHS = cfg["paths"]
ANALYSIS = cfg["analysis"]
nii_str = ANALYSIS["nii_str"]
first_level_path = resolve_config_path(PATHS["first_level_derivatives"]) / nii_str
mask_paths = str(resolve_config_path(PATHS["brain_mask"]))

nii_files = []
for subdir in sorted(str(p) for p in first_level_path.glob("sub-*/patterns")):
    sub_nii_files = glob(op.join(subdir, "*-*_beta.nii.gz"))
    if len(sub_nii_files) > 1:
        sub_niis = [f for f in sub_nii_files if "error_trial" not in f]
    else:
        sub_niis = sub_nii_files
    nii_files.extend(sub_niis)

df = pd.read_csv(artifacts_dir / "design_df.csv")
df_factor = pd.read_csv(artifacts_dir / "design_factors.csv")
df_dum = pd.read_csv(artifacts_dir / "design_subject_dummies.csv")

imgs = [nib.load(fp) for fp in nii_files]
mask = nib.load(mask_paths)

memory = Memory(location=str(cache_dir), verbose=0)

print(f"Reloaded {len(imgs)} maps for {df['subject'].nunique()} subjects.")


Reloaded 24 maps for 3 subjects.


In [2]:
def custom_contrast(levels):
    if len(levels) == 1:
        contrast_matrix = [
            [1],[-1]]

    elif len(levels) == 2:
        contrast_matrix = [
            [1, 0],
            [0, 1],
            [-1, -1]
        ]
    return ContrastMatrix(contrast_matrix, levels)



def get_dm(df_factor, df_dum=None):
    """
    Compute the behavioral design matrix and behavioral RDMs.
    Outputs:
        X: design matrix
        beh_dm: vectors of behavioral RDMs
    """

    factors = df_factor.columns.tolist()
    contrast2 ={}
    contrast3 ={}
    formula = ''
    for i, factor in enumerate(factors):
    # Define the model formula
        if np.unique(df_factor[factor]).size == 2:
            contrast2[factor] = [f'{np.unique(df_factor[factor])[0]}-{np.unique(df_factor[factor])[1]}']
            if factor != factors[-1]:
                formula += f'C({factor}, contrast=custom_contrast(contrast2["{factor}"])) *'
            else:
                formula += f'C({factor}, contrast=custom_contrast(contrast2["{factor}"]))'

        elif np.unique(df_factor[factor]).size == 3:
            contrast3[factor] = [f'{np.unique(df_factor[factor])[0]}-{np.unique(df_factor[factor])[2]}', f'{np.unique(df_factor[factor])[1]}-{np.unique(df_factor[factor])[2]}']
            if factor != factors[-1]:
                formula += f'C({factor}, contrast=custom_contrast(contrast3["{factor}"])) *'
            else:
                formula += f'C({factor}, contrast=custom_contrast(contrast3["{factor}"]))'
    formula = formula[:-1] if formula.endswith('*') else formula            
    X = patsy.dmatrix(formula, df_factor, return_type='dataframe')

    pattern = r"\)\)([a-zA-Z0-9_]+-[a-zA-Z0-9_]+)"

    # Use re.findall() to find all matches
    matches = [re.findall(pattern, column) for column in X.columns[1:].tolist()]
    beh_names = ["_x_".join(match) for match in matches]
    print(beh_names)
    beh_names.insert(0, "intercept")
    X.columns = beh_names
    print('X:', X)
    df_numeric_to_str = df_factor.select_dtypes(include=['number']).astype(str)
    # all number to string, get_dummies() can't handle numeric columns    
    df_factor[df_numeric_to_str.columns] = df_numeric_to_str
    dum_factor = pd.get_dummies(df_factor.iloc[:,:len(factors)])
    if df_dum is not None:
        df_numeric_to_str = df_dum.select_dtypes(include=['number']).astype(str)
        df_dum[df_numeric_to_str.columns] = df_numeric_to_str
        dum_dum = pd.get_dummies(df_dum)
        # dum_dum = pd.concat([dum_dum, df_dum['run_num']], axis=1)
    else:
        dum_dum = None
    print('dum_factor:', dum_factor)
    return X, dum_factor, dum_dum


In [3]:
def main_interaction_effects(columns, factors):
    effects = {}
    processed_factors = set()  # Track factors that have already been processed

    for col in columns:
        parts = col.split('_x_')
        if len(parts) == 1:
            # Main effect
            for factor, levels in factors.items():
                if factor in processed_factors:
                    continue  # Skip factors that have already been processed

                if any(level in parts[0] for level in levels):
                    effects[col] = factor
                    processed_factors.add(factor)  # Mark this factor as processed
                    break
        else:
            # Interaction
            interaction_factors = []
            for part in parts:
                for factor, levels in factors.items():
                    if factor in processed_factors:
                        continue  # Skip factors that have already been processed

                    if any(level in part for level in levels):
                        interaction_factors.append(factor)
                        processed_factors.add(factor)  # Mark this factor as processed
                        break
            effects[col] = '_x_'.join(interaction_factors)

    return effects


def glm_model(df, df_dum=None, smooth = None, njobs = -1):

    try:
        X, dum_factors, dum_dum = get_dm(df, df_dum)
    except Exception as e:
        raise ValueError("Failed to build the design matrix. Check the input data.") from e

    columns = X.columns.tolist()
    X_neg_values = X.values * -1
    X_neg_columns = [f"neg_{col}" for col in columns]
    X_neg = pd.DataFrame(X_neg_values, columns=X_neg_columns, index=X.index)
    # Define factors and their levels
    # Ensure all levels in factors are strings
    factors = {col: df[col].unique().astype(str).tolist() for col in df.columns}


    # Auto-generate the effects dictionary
    effects = main_interaction_effects(columns, factors)

    # Ensure effects is not empty
    if not effects:
        raise ValueError("No effects dictionary found. Check the input data.")

    # Build a dictionary to store the position of each effect
    effect_positions = {col: i for i, col in enumerate(columns)}

    # Group columns by effect
    grouped_effects = {}
    for col, effect in effects.items():
        grouped_effects.setdefault(effect, []).append(col)

    # Initialize contrast_dicts
    contrast_dicts = {}
    if dum_dum is not None:
        dum_num = dum_dum.shape[1]
    else:
        dum_num = 0
    # Populate contrast_dicts for main contrasts (F test, multiple contrasts)
    for effect, cols in grouped_effects.items():
        contrast_dict = np.zeros((len(cols), len(columns) + dum_num))

        for i, col in enumerate(cols):
            position = effect_positions[col]
            contrast_dict[i, position] = 1

        # Add positive and negative contrasts to the dictionary
        contrast_dicts[effect] = contrast_dict
        # contrast_dicts[f"-{effect}"] = np.where(contrast_dicts, -1, 0)

    # Print the final contrast dictionary
    print("Final contrast_dicts:")
    for effect_name, contrast_dict in contrast_dicts.items():
        print(f"Effect: {effect_name}")
        print(contrast_dict)
        print()

    # For simple contrasts (t test, single contrast)
    simple_contrasts = {}
    for col in columns:
        simple_contrast = np.zeros(len(columns) + dum_num)
        position = effect_positions[col]
        simple_contrast[position] = 1
        col = col if col not in simple_contrasts.keys() else f"{col}_2"
        simple_contrasts[col] = simple_contrast
        # simple_contrasts[f"-{col}"] = np.where(simple_contrast, -1, 0)

    # Print simple contrasts
    print("Final simple_contrasts:")
    for effect_name, simple_contrast in simple_contrasts.items():
        print(f"Effect: {effect_name}")
        print(simple_contrast)
        print()

    design_matrix = pd.concat([X, dum_dum.astype(int)], axis=1)
    design_matrix_neg = pd.concat([X_neg, dum_dum.astype(int)], axis=1)
    glm = SecondLevelModel(
        memory=memory, memory_level=2, minimize_memory=True,
        smoothing_fwhm=smooth, n_jobs=njobs, mask_img=mask
    ).fit(
        imgs,
        design_matrix=design_matrix,
    )


    return glm, contrast_dicts, simple_contrasts

glm, contrast_dicts, simple_contrasts = glm_model(df=df_factor, df_dum=df_dum, smooth = None, njobs = -1)


['Long-Short', 'diff-same', 'flexibility-stability', 'Long-Short_x_diff-same', 'Long-Short_x_flexibility-stability', 'diff-same_x_flexibility-stability', 'Long-Short_x_diff-same_x_flexibility-stability']
X:     intercept  Long-Short  diff-same  flexibility-stability  \
0         1.0         1.0        1.0                    1.0   
1         1.0         1.0        1.0                   -1.0   
2         1.0         1.0       -1.0                    1.0   
3         1.0         1.0       -1.0                   -1.0   
4         1.0        -1.0        1.0                    1.0   
5         1.0        -1.0        1.0                   -1.0   
6         1.0        -1.0       -1.0                    1.0   
7         1.0        -1.0       -1.0                   -1.0   
8         1.0         1.0        1.0                    1.0   
9         1.0         1.0        1.0                   -1.0   
10        1.0         1.0       -1.0                    1.0   
11        1.0         1.0       -1.0 

In [4]:
def contrast_effects(contrast_key, contrast_value, glm):
    print(f"Contrast: {contrast_key}")

    if len(contrast_value.shape)==1 or contrast_value.shape[0] == 1:
        z_map = glm.compute_contrast(
            second_level_stat_type='t',
            second_level_contrast=contrast_value,
            output_type='z_score'
        )
    elif len(contrast_value.shape)==2 and contrast_value.shape[0] > 1:
        z_map = glm.compute_contrast(
            second_level_stat_type='F',
            second_level_contrast=contrast_value,
            output_type='z_score'
        )
    else:
        raise ValueError("Contrast values should be a 1D or 2D array.")

    # Create output directory
    output_dir = os.path.join(str(res_path), nii_str, f'{contrast_key}_contrast')
    if not op.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)
    z_map.to_filename(f'{output_dir}/{contrast_key}_unthresholded.nii')
    # print(f"Report generated and saved to {output_dir}")

    return {
        'z_map': z_map
        # Other result fields can be added here...
    }


In [5]:
effect = ["simple"]
if "main" in effect:
    results = Parallel(n_jobs=cfg["compute"]["n_jobs_contrasts"])(
        delayed(contrast_effects)(contrast_key, contrast_value, glm)
        for contrast_key, contrast_value in contrast_dicts.items()
    )
elif "simple" in effect:
    simple_results = Parallel(n_jobs=cfg["compute"]["n_jobs_contrasts"])(
        delayed(contrast_effects)(simple_key, simple_value, glm)
        for simple_key, simple_value in simple_contrasts.items()
    )

print("Unthresholded z-maps written under:", Path(res_path) / nii_str)


Contrast: intercept
Contrast: Long-Short
Contrast: diff-same
Contrast: flexibility-stability
Contrast: Long-Short_x_diff-same
Contrast: Long-Short_x_flexibility-stability


Contrast: diff-same_x_flexibility-stability


Contrast: Long-Short_x_diff-same_x_flexibility-stability
Unthresholded z-maps written under: D:\文件同步\文件\数据分析结果\data_code_figure\brain_activation\results\second_level\full_factorial
